## Read in data

In [ ]:
"""
敏感度分析脚本
每次修改一个参数，分别保存去噪和整合的结果到不同的CSV文件
"""

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import sys
import os
from sklearn.metrics import (
    normalized_mutual_info_score, mutual_info_score, adjusted_mutual_info_score,
    v_measure_score, homogeneity_score, completeness_score,
    adjusted_rand_score, fowlkes_mallows_score
)

# 导入CANDIES相关模块（从根目录的codes文件夹导入）
from codes1.DiTs import seed_everything
from codes1.sampler import *
from codes1.train_diff import *
from codes1.ZINB_encoder import *
from codes1.preprocess1 import *
from codes1.AutoEncoder import *
from codes1.get_graph import *
from codes1.integration import *


# 默认参数值
DEFAULT_SPATIAL_K = 3
DEFAULT_FEATURE_K = 20
DEFAULT_DIFFUSION_STEP = 800
DEFAULT_ALPHA = 2.0
DEFAULT_BETA = 0.1
DEFAULT_LAMBDA_CL = 0.5  # 对比学习损失权重
DEFAULT_LAMBDA_REC = 0.5  # 重构损失权重（lambda_cl + lambda_rec = 1）

# 参数测试范围（每次只修改一个参数，其他保持默认值）
PARAM_RANGES = {
    'spatial_k': [1, 3, 5, 8, 10],  # 空间图的k值
    'feature_k': [10, 15, 20, 25, 30],  # 特征图的k值
    'diffusion_step': [300, 500, 800, 1000, 1200],  # Diffusion步数
    'alpha': [1.0, 1.5, 2.0, 2.5, 3.0],  # 编码阶段重构损失权重
    'beta': [0.05, 0.1, 0.15, 0.2, 0.3],  # 编码阶段ZINB损失权重
    'lambda_cl': [0.1, 0.3, 0.5, 0.7, 0.9]  # 整合阶段对比学习损失权重（lambda_rec = 1 - lambda_cl）
}

# 是否运行所有参数组合（True: 遍历所有参数范围, False: 只运行默认值）
RUN_ALL_PARAMS = True

# 数据路径
file_fold = 'E:/yan0/ours/data/simulation/'

# 结果保存路径
RESULT_DIR = "E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results"
os.makedirs(RESULT_DIR, exist_ok=True)


# ============================================================================
# 辅助函数
# ============================================================================

class ConditionalDiffusionDataset():
    def __init__(self, adata_omics1, adata_omics2):
        self.adata_omics1 = adata_omics1
        self.adata_omics2 = adata_omics2
        self.st_sample = torch.tensor(self.adata_omics1, dtype=torch.float32)
        self.con_sample = torch.tensor(self.adata_omics2, dtype=torch.float32)
        self.con_data = torch.tensor(self.adata_omics2, dtype=torch.float32)

    def __len__(self):
        return len(self.adata_omics1)

    def __getitem__(self, idx):
        return self.st_sample[idx], self.con_sample[idx], self.con_data


def calculate_metrics(ground_truth, predictions):
    """计算所有评估指标"""
    metrics = {
        'Mutual Information': mutual_info_score(ground_truth, predictions),
        'NMI': normalized_mutual_info_score(ground_truth, predictions),
        'AMI': adjusted_mutual_info_score(ground_truth, predictions),
        'V-measure': v_measure_score(ground_truth, predictions),
        'Homogeneity': homogeneity_score(ground_truth, predictions),
        'Completeness': completeness_score(ground_truth, predictions),
        'ARI': adjusted_rand_score(ground_truth, predictions),
        'FMI': fowlkes_mallows_score(ground_truth, predictions)
    }
    return metrics


def print_metrics(metrics, method_name):
    """打印评估指标"""
    print(f'\n{method_name}\n')
    for key, value in metrics.items():
        print(f"{key}: {value:.6f}")


# ============================================================================
# 主流程
# ============================================================================


def run_single_experiment(adata_omics1, adata_omics2, spatial_k, feature_k, diffusion_step, alpha, beta, lambda_cl, lambda_rec):
    """运行单次实验"""
    # 识别当前修改的参数类型（用于生成文件名）
    param_type = None
    param_value = None
    if spatial_k != DEFAULT_SPATIAL_K:
        param_type = "spatial_k"
        param_value = spatial_k
    elif feature_k != DEFAULT_FEATURE_K:
        param_type = "feature_k"
        param_value = feature_k
    elif diffusion_step != DEFAULT_DIFFUSION_STEP:
        param_type = "diffusion_step"
        param_value = diffusion_step
    elif alpha != DEFAULT_ALPHA:
        param_type = "alpha"
        param_value = alpha
    elif beta != DEFAULT_BETA:
        param_type = "beta"
        param_value = beta
    elif lambda_cl != DEFAULT_LAMBDA_CL:
        param_type = "lambda_cl"
        param_value = lambda_cl
    else:
        param_type = "default"
        param_value = "default"
    
    # 生成参数名称（用于显示）
    param_name_parts = []
    if spatial_k != DEFAULT_SPATIAL_K:
        param_name_parts.append(f"spatial_k_{spatial_k}")
    if feature_k != DEFAULT_FEATURE_K:
        param_name_parts.append(f"feature_k_{feature_k}")
    if diffusion_step != DEFAULT_DIFFUSION_STEP:
        param_name_parts.append(f"diffusion_step_{diffusion_step}")
    if alpha != DEFAULT_ALPHA:
        param_name_parts.append(f"alpha_{alpha}")
    if beta != DEFAULT_BETA:
        param_name_parts.append(f"beta_{beta}")
    if lambda_cl != DEFAULT_LAMBDA_CL:
        param_name_parts.append(f"lambda_cl_{lambda_cl}")
    
    if not param_name_parts:
        current_param_name = "default"
    else:
        current_param_name = "_".join(param_name_parts)
    
    print("=" * 80)
    print(f"开始敏感度分析 - 参数: {current_param_name}")
    print(f"  spatial_k={spatial_k}, feature_k={feature_k}, diffusion_step={diffusion_step}")
    print(f"  alpha={alpha}, beta={beta}, lambda_cl={lambda_cl}, lambda_rec={lambda_rec}")
    print("=" * 80)

    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print(f"使用设备: {device}")
    
    # 复制数据，避免修改原始数据
    adata_omics1 = adata_omics1.copy()
    adata_omics2 = adata_omics2.copy()
    
    # ========================================================================
    # 1. 构建图（使用参数化的spatial_k和feature_k）
    # ========================================================================
    print(f"\n[1/5] 构建图 (spatial_k={spatial_k}, feature_k={feature_k})...")
    adata_omics1, adata_omics2 = construct_neighbor_graph(
        adata_omics1, adata_omics2,
        spatial_k=spatial_k,
        feature_k=feature_k
    )
    
    adj = adjacent_matrix_preprocessing(adata_omics1, adata_omics2)
    adj_spatial_omics1 = adj['adj_spatial_omics1'].to(device)
    adj_spatial_omics2 = adj['adj_spatial_omics2'].to(device)
    
    # ========================================================================
    # 2. 编码阶段（使用参数化的alpha和beta）
    # ========================================================================
    print(f"\n[2/5] 编码阶段 (alpha={alpha}, beta={beta})...")
    ae_model = encoder_ZINB(
        adata=adata_omics1,
        device=torch.device(device),
        epochs=300,
        dim_output=128,
        alpha=alpha,
        beta=beta
    )
    adata_omics1.obsm['emb_ZINB'], adj_mat = ae_model.train()
    
    seed = 2024
    seed_everything(seed)
    train_model(adata_omics1, adata_omics2, adj_spatial_omics1, adj_spatial_omics2, epochs=200)
    
    # ========================================================================
    # 3. 去噪阶段（使用参数化的diffusion_step）
    # ========================================================================
    print(f"\n[3/5] 去噪阶段 (diffusion_step={diffusion_step})...")
    # 对齐embeddings
    slices_omics1_spatial = adata_omics1.obsm['spatial']
    slices_omics2_spatial = adata_omics2.obsm['spatial']
    emb_latent_omics1 = adata_omics1.obsm['emb_ZINB']
    emb_latent_omics2 = adata_omics2.obsm['emb_latent_omics2']
    
    df_omics1 = pd.DataFrame(emb_latent_omics1, index=[tuple(coord) for coord in slices_omics1_spatial])
    df_omics2 = pd.DataFrame(emb_latent_omics2, index=[tuple(coord) for coord in slices_omics2_spatial])
    df_omics2_aligned = df_omics2.reindex(df_omics1.index)
    aligned_emb_latent_omics1 = df_omics1.to_numpy()
    aligned_emb_latent_omics2 = df_omics2_aligned.to_numpy()
    
    # Diffusion
    dataset = ConditionalDiffusionDataset(aligned_emb_latent_omics1, aligned_emb_latent_omics2)
    seed = 42
    seed_everything(seed)
    
    com_mtx = run_diff(
        dataset,
        k=3,
        batch_size=512,
        hidden_size=512,
        learning_rate=1e-3,
        num_epoch=1000,
        diffusion_step=diffusion_step,  # 使用参数
        depth=6,
        head=16,
        device=device,
        classes=6,
        patience=40,
        bias=1
    )
    
    adata_omics1.obsm['denoise_emb'] = com_mtx
    
    # 聚类和评估
    tool = 'mclust'
    clustering(adata_omics1, key='denoise_emb', add_key='Denoise', n_clusters=5, end=1.2, method=tool, use_pca=True)
    
    # 计算去噪阶段的指标
    denoise_metrics = calculate_metrics(adata_omics1.obs['ground_truth'], adata_omics1.obs['Denoise'])
    print_metrics(denoise_metrics, '去噪结果 (ADT_denoised)')
    
    # 保存去噪结果（追加到同一类参数的文件中）
    denoise_results = {
        'Parameter': [current_param_name],
        'spatial_k': [spatial_k],
        'feature_k': [feature_k],
        'diffusion_step': [diffusion_step],
        'alpha': [alpha],
        'beta': [beta],
        'lambda_cl': [lambda_cl],
        'lambda_rec': [lambda_rec],
        **denoise_metrics
    }
    denoise_df = pd.DataFrame(denoise_results)
    denoise_csv_file = os.path.join(RESULT_DIR, f"denoise_results_{param_type}.csv")
    
    # 追加模式：如果文件存在则追加，否则创建新文件
    if os.path.exists(denoise_csv_file):
        denoise_df.to_csv(denoise_csv_file, mode='a', header=False, index=False)
        print(f"去噪结果已追加到: {denoise_csv_file}")
    else:
        denoise_df.to_csv(denoise_csv_file, index=False)
        print(f"去噪结果已保存到: {denoise_csv_file}")
    
    # ========================================================================
    # 4. 整合阶段（使用参数化的lambda_cl和lambda_rec）
    # ========================================================================
    print(f"\n[4/5] 整合阶段 (lambda_cl={lambda_cl}, lambda_rec={lambda_rec})...")
    adata1 = adata_omics1.copy()
    adata2 = adata_omics2.copy()
    
    adata1.obsm['feat'] = adata1.obsm['denoise_emb']
    adata2.obsm['feat'] = adata2.obsm['emb_latent_omics2']
    adata1, adata2 = construct_neighbor_graph(adata1, adata2, spatial_k=spatial_k, feature_k=feature_k)
    
    adj = adjacent_matrix_preprocessing(adata1, adata2)
    adj_spatial_omics1 = adj['adj_spatial_omics1'].to(device)
    adj_spatial_omics2 = adj['adj_spatial_omics2'].to(device)
    adj_feature_omics1 = adj['adj_feature_omics1'].to(device)
    adj_feature_omics2 = adj['adj_feature_omics2'].to(device)
    
    features_omics1 = torch.FloatTensor(adata1.obsm['feat'].copy()).to(device)
    features_omics2 = torch.FloatTensor(adata2.obsm['feat'].copy()).to(device)
    
    seed = 2025
    seed_everything(seed)
    
    result = train_and_infer(
        features_omics1=features_omics1,
        features_omics2=features_omics2,
        adj_spatial_omics1=adj_spatial_omics1,
        adj_feature_omics1=adj_feature_omics1,
        adj_spatial_omics2=adj_spatial_omics2,
        adj_feature_omics2=adj_feature_omics2,
        device=device,
        epochs=100,
        lambda_cl=lambda_cl,  # 使用参数
        lambda_rec=lambda_rec  # 使用参数
    )
    
    adata = adata1.copy()
    candies_emb = result['emb_latent_combined'].detach().cpu().numpy().copy()
    adata.obsm['CANDIES'] = candies_emb
    
    # 聚类和评估
    tool = 'mclust'
    clustering(adata, key='CANDIES', add_key='CANDIES', n_clusters=5, end=1, method=tool, use_pca=True)
    
    # 计算整合阶段的指标
    integration_metrics = calculate_metrics(adata.obs['ground_truth'], adata.obs['CANDIES'])
    print_metrics(integration_metrics, '整合结果 (CANDIES)')
    
    # 保存整合结果（追加到同一类参数的文件中）
    integration_results = {
        'Parameter': [current_param_name],
        'spatial_k': [spatial_k],
        'feature_k': [feature_k],
        'diffusion_step': [diffusion_step],
        'alpha': [alpha],
        'beta': [beta],
        'lambda_cl': [lambda_cl],
        'lambda_rec': [lambda_rec],
        'Method': ['CANDIES(ADT_denoise)'],
        **integration_metrics
    }
    integration_df = pd.DataFrame(integration_results)
    integration_csv_file = os.path.join(RESULT_DIR, f"integration_results_{param_type}.csv")
    
    # 追加模式：如果文件存在则追加，否则创建新文件
    if os.path.exists(integration_csv_file):
        integration_df.to_csv(integration_csv_file, mode='a', header=False, index=False)
        print(f"整合结果已追加到: {integration_csv_file}")
    else:
        integration_df.to_csv(integration_csv_file, index=False)
        print(f"整合结果已保存到: {integration_csv_file}")
    
    # ========================================================================
    # 5. 完成
    # ========================================================================
    print("\n" + "=" * 80)
    print(f"敏感度分析完成 - 参数: {current_param_name}")
    print("=" * 80)
    print(f"\n结果文件:")
    print(f"  去噪结果: {denoise_csv_file}")
    print(f"  整合结果: {integration_csv_file}")
    
    return current_param_name


In [3]:
import numpy as np
import scanpy as sc

# read data
file_fold = 'E:/yan0/ours/data/simulation/'  #please replace 'file_fold' with the download path

adata_omics1 = sc.read_h5ad(file_fold + 'rna(std3).h5ad')
adata_omics2 = sc.read_h5ad(file_fold + 'adt(std1).h5ad')

adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()

In [4]:
import pandas as pd

# 提取空间坐标
spatial1 = pd.DataFrame(adata_omics1.obsm['spatial'], columns=['x', 'y'])
spatial2 = pd.DataFrame(adata_omics2.obsm['spatial'], columns=['x', 'y'])

# 为两个数据集的空间坐标添加索引
spatial1['index1'] = spatial1.index
spatial2['index2'] = spatial2.index

# 根据空间坐标合并两个数据集，确保顺序一致
merged = pd.merge(spatial1, spatial2, on=['x', 'y'], how='inner')

# 获取重新排序后的索引
sorted_index1 = merged['index1'].values
sorted_index2 = merged['index2'].values

# 重新排序原始数据
adata_omics1 = adata_omics1[sorted_index1]
adata_omics2 = adata_omics2[sorted_index2]

labels = pd.read_csv(file_fold + 'simulation_ground_truth.csv')
labels.index = adata_omics1.obs.index

adata_omics1.obs['ground_truth'] = labels['leiden1']
adata_omics2.obs['ground_truth'] = labels['leiden1']

In [5]:
def run_leiden(adata1, n_cluster, use_rep="embeddings", key_added="Nleiden", range_min=0, range_max=3, max_steps=30, tolerance=0):
    adata = adata1.copy()
    sc.pp.neighbors(adata, use_rep=use_rep)
    this_step = 0
    this_min = float(range_min)
    this_max = float(range_max)
    while this_step < max_steps:
        this_resolution = this_min + ((this_max-this_min)/2)
        sc.tl.leiden(adata, resolution=this_resolution)
        this_clusters = adata.obs['leiden'].nunique()

        if this_clusters > n_cluster+tolerance:
            this_max = this_resolution
        elif this_clusters < n_cluster-tolerance:
            this_min = this_resolution
        else:
            print("Succeed to find %d clusters at resolution %.3f"%(n_cluster, this_resolution))
            adata1.obs[key_added] = adata.obs["leiden"]
            
            return adata1
        
        this_step += 1
    
    adata1.obs[key_added] = adata.obs["leiden"]
    return adata1

In [6]:
import sys
import os
# 添加项目根目录到 Python 路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from codes_v2.DiTs import *
from codes_v2.sampler import *
from codes_v2.train_diff import *
from codes_v2.ZINB_encoder import *
from codes_v2.preprocess1 import *

e:\yan0\ours\CANDIES_v2\sensitivity_analysis
NVIDIA GeForce RTX 4060


## Data preprocessing

In [5]:
# RNA
sc.pp.filter_genes(adata_omics1, min_cells=50)
sc.pp.filter_genes(adata_omics1, min_counts=10)
sc.pp.normalize_total(adata_omics1, target_sum=1e6)
sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=2000)
sc.pp.scale(adata_omics1)

adata_omics1_high =  adata_omics1[:, adata_omics1.var['highly_variable']]
adata_omics1.obsm['feat'] = pca(adata_omics1_high, n_comps=adata_omics2.n_vars-1)

# Protein
sc.pp.scale(adata_omics2)
adata_omics2.obsm['feat'] = pca(adata_omics2, n_comps=adata_omics2.n_vars-1)

## Encoding phase

In [ ]:
def main():
    """主函数：遍历所有参数组合"""
    print("=" * 80)
    print("敏感度分析 - 批量运行所有参数组合")
    print("=" * 80)
    
    if not RUN_ALL_PARAMS:
        # 只运行默认参数
        print("\n只运行默认参数...")
        run_single_experiment(
            adata_omics1, adata_omics2,
            DEFAULT_SPATIAL_K, DEFAULT_FEATURE_K, DEFAULT_DIFFUSION_STEP,
            DEFAULT_ALPHA, DEFAULT_BETA, DEFAULT_LAMBDA_CL, DEFAULT_LAMBDA_REC
        )
    else:
        # 遍历所有参数（每次只改一个参数）
        total_experiments = sum(len(v) for v in PARAM_RANGES.values()) - len(PARAM_RANGES) + 1  # 减去重复的默认值
        current_exp = 0
        
        # 2. 遍历 spatial_k
        for spatial_k in PARAM_RANGES['spatial_k']:
            if spatial_k == DEFAULT_SPATIAL_K:
                continue
            current_exp += 1
            print(f"\n[{current_exp}/{total_experiments}] 测试 spatial_k={spatial_k}...")
            run_single_experiment(
                adata_omics1, adata_omics2,
                spatial_k, DEFAULT_FEATURE_K, DEFAULT_DIFFUSION_STEP,
                DEFAULT_ALPHA, DEFAULT_BETA, DEFAULT_LAMBDA_CL, DEFAULT_LAMBDA_REC
            )
        
        # 3. 遍历 feature_k
        for feature_k in PARAM_RANGES['feature_k']:
            if feature_k == DEFAULT_FEATURE_K:
                continue
            current_exp += 1
            print(f"\n[{current_exp}/{total_experiments}] 测试 feature_k={feature_k}...")
            run_single_experiment(
                adata_omics1, adata_omics2,
                DEFAULT_SPATIAL_K, feature_k, DEFAULT_DIFFUSION_STEP,
                DEFAULT_ALPHA, DEFAULT_BETA, DEFAULT_LAMBDA_CL, DEFAULT_LAMBDA_REC
            )
        
        # 4. 遍历 diffusion_step
        for diffusion_step in PARAM_RANGES['diffusion_step']:
            if diffusion_step == DEFAULT_DIFFUSION_STEP:
                continue
            current_exp += 1
            print(f"\n[{current_exp}/{total_experiments}] 测试 diffusion_step={diffusion_step}...")
            run_single_experiment(
                adata_omics1, adata_omics2,
                DEFAULT_SPATIAL_K, DEFAULT_FEATURE_K, diffusion_step,
                DEFAULT_ALPHA, DEFAULT_BETA, DEFAULT_LAMBDA_CL, DEFAULT_LAMBDA_REC
            )
        
        # 5. 遍历 alpha
        for alpha in PARAM_RANGES['alpha']:
            if alpha == DEFAULT_ALPHA:
                continue
            current_exp += 1
            print(f"\n[{current_exp}/{total_experiments}] 测试 alpha={alpha}...")
            run_single_experiment(
                adata_omics1, adata_omics2,
                DEFAULT_SPATIAL_K, DEFAULT_FEATURE_K, DEFAULT_DIFFUSION_STEP,
                alpha, DEFAULT_BETA, DEFAULT_LAMBDA_CL, DEFAULT_LAMBDA_REC
            )
        
        # 6. 遍历 beta
        for beta in PARAM_RANGES['beta']:
            if beta == DEFAULT_BETA:
                continue
            current_exp += 1
            print(f"\n[{current_exp}/{total_experiments}] 测试 beta={beta}...")
            run_single_experiment(
                adata_omics1, adata_omics2,
                DEFAULT_SPATIAL_K, DEFAULT_FEATURE_K, DEFAULT_DIFFUSION_STEP,
                DEFAULT_ALPHA, beta, DEFAULT_LAMBDA_CL, DEFAULT_LAMBDA_REC
            )
        
        # 7. 遍历 lambda_cl（lambda_rec自动计算为1-lambda_cl）
        for lambda_cl in PARAM_RANGES['lambda_cl']:
            if lambda_cl == DEFAULT_LAMBDA_CL:
                continue
            lambda_rec = 1.0 - lambda_cl
            current_exp += 1
            print(f"\n[{current_exp}/{total_experiments}] 测试 lambda_cl={lambda_cl}, lambda_rec={lambda_rec}...")
            run_single_experiment(
                adata_omics1, adata_omics2,
                DEFAULT_SPATIAL_K, DEFAULT_FEATURE_K, DEFAULT_DIFFUSION_STEP,
                DEFAULT_ALPHA, DEFAULT_BETA, lambda_cl, lambda_rec
            )
        
        print("\n" + "=" * 80)
        print(f"所有实验完成！共运行 {current_exp} 个参数组合")
        print("=" * 80)


if __name__ == "__main__":
    main()


敏感度分析 - 批量运行所有参数组合

[1/25] 测试 spatial_k=1...
开始敏感度分析 - 参数: spatial_k_1
  spatial_k=1, feature_k=20, diffusion_step=800
  alpha=2.0, beta=0.1, lambda_cl=0.5, lambda_rec=0.5
使用设备: cuda:0

[1/5] 构建图 (spatial_k=1, feature_k=20)...

[2/5] 编码阶段 (alpha=2.0, beta=0.1)...


100%|██████████| 300/300 [00:01<00:00, 206.79it/s]


Optimization finished
Epoch [100/200], Loss RNA: 1.5608, Loss Protein: 0.5922
Epoch [200/200], Loss RNA: 1.3560, Loss Protein: 0.5013
Training complete!
Latent representations have been successfully added to adata_omics.obsm!

[3/5] 去噪阶段 (diffusion_step=800)...
*******************  Fold 1  (total 3 folds)  ********************


 44%|██████████▉              | 437/1000 [02:01<02:36,  3.59it/s, noise loss:0.0342539, lr:1.00e-07]


Early stop(patience:40)!!


...

 37%|█████████▏               | 366/1000 [01:37<02:49,  3.74it/s, noise loss:0.0371149, lr:1.00e-06]


Early stop(patience:40)!!


time: 0: 100%|██████████| 800/800 [00:50<00:00, 15.95it/s]  


*******************  Fold 3  (total 3 folds)  ********************


 25%|██████▏                  | 249/1000 [01:16<03:49,  3.27it/s, noise loss:0.0538289, lr:1.00e-05]

In [4]:
"""
添加默认参数结果到所有CSV文件
"""
import pandas as pd
import os

# 结果目录
script_dir = 'E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results'
result_dir = script_dir

# 默认参数值
DEFAULT_SPATIAL_K = 3
DEFAULT_FEATURE_K = 20
DEFAULT_DIFFUSION_STEP = 800
DEFAULT_ALPHA = 2.0
DEFAULT_BETA = 0.1
DEFAULT_LAMBDA_CL = 0.5
DEFAULT_LAMBDA_REC = 0.5

# 图一：去噪默认参数结果
denoise_default = {
    'Parameter': 'default',
    'spatial_k': DEFAULT_SPATIAL_K,
    'feature_k': DEFAULT_FEATURE_K,
    'diffusion_step': DEFAULT_DIFFUSION_STEP,
    'alpha': DEFAULT_ALPHA,
    'beta': DEFAULT_BETA,
    'lambda_cl': DEFAULT_LAMBDA_CL,
    'lambda_rec': DEFAULT_LAMBDA_REC,
    'Mutual Information': 1.389215,
    'NMI': 0.896195,
    'AMI': 0.895778,
    'V-measure': 0.896195,
    'Homogeneity': 0.897759,
    'Completeness': 0.894638,
    'ARI': 0.908379,
    'FMI': 0.929012
}

# 图二：整合默认参数结果
integration_default = {
    'Parameter': 'default',
    'spatial_k': DEFAULT_SPATIAL_K,
    'feature_k': DEFAULT_FEATURE_K,
    'diffusion_step': DEFAULT_DIFFUSION_STEP,
    'alpha': DEFAULT_ALPHA,
    'beta': DEFAULT_BETA,
    'lambda_cl': DEFAULT_LAMBDA_CL,
    'lambda_rec': DEFAULT_LAMBDA_REC,
    'Method': 'CANDIES(ADT_denoise)',
    'Mutual Information': 1.440995,
    'NMI': 0.93021,
    'AMI': 0.929929,
    'V-measure': 0.93021,
    'Homogeneity': 0.93122,
    'Completeness': 0.929202,
    'ARI': 0.943354,
    'FMI': 0.95613
}

# 所有参数类型
param_keys = ['spatial_k', 'feature_k', 'diffusion_step', 'alpha', 'beta', 'lambda_cl']

# 更新去噪结果文件
for param_key in param_keys:
    denoise_file = os.path.join(result_dir, f"denoise_results_{param_key}.csv")
    if os.path.exists(denoise_file):
        df = pd.read_csv(denoise_file)
        # 检查是否已有默认参数行
        if 'default' not in df['Parameter'].values:
            # 添加默认参数行
            new_row = pd.DataFrame([denoise_default])
            df = pd.concat([df, new_row], ignore_index=True)
            df.to_csv(denoise_file, index=False)
            print(f"已添加默认参数到: {denoise_file}")

# 更新整合结果文件
for param_key in param_keys:
    integration_file = os.path.join(result_dir, f"integration_results_{param_key}.csv")
    if os.path.exists(integration_file):
        df = pd.read_csv(integration_file)
        # 检查是否已有默认参数行
        if 'default' not in df['Parameter'].values:
            # 添加默认参数行
            new_row = pd.DataFrame([integration_default])
            df = pd.concat([df, new_row], ignore_index=True)
            df.to_csv(integration_file, index=False)
            print(f"已添加默认参数到: {integration_file}")

print("\n所有CSV文件更新完成！")


已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\denoise_results_spatial_k.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\denoise_results_feature_k.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\denoise_results_diffusion_step.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\denoise_results_alpha.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\denoise_results_beta.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\denoise_results_lambda_cl.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\integration_results_spatial_k.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\integration_results_feature_k.csv
已添加默认参数到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\integrati

In [10]:
"""
绘制敏感度分析结果折线图
横轴：参数值，纵轴：ARI
去噪和整合分开绘制，每个参数生成两个PNG图片
"""
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# 设置中文字体（如果系统支持）
try:
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
except:
    pass
plt.rcParams['axes.unicode_minus'] = False

# 结果目录
script_dir = 'E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results'
result_dir = script_dir
figs_dir = os.path.join(script_dir, "figs")
os.makedirs(figs_dir, exist_ok=True)

# 参数配置：参数键 -> (参数列名, 参数显示名称)
param_configs = {
    "spatial_k": ("spatial_k", "Spatial k"),
    "feature_k": ("feature_k", "Feature k"),
    "diffusion_step": ("diffusion_step", "Diffusion Step"),
    "alpha": ("alpha", "Alpha"),
    "beta": ("beta", "Beta"),
    "lambda_cl": ("lambda_cl", "Lambda CL")
}

# 为每个参数分别绘制去噪和整合的图
for param_key, (param_col, param_label) in param_configs.items():
    # 读取去噪结果
    denoise_file = os.path.join(result_dir, f"denoise_results_{param_key}.csv")
    # 读取整合结果
    integration_file = os.path.join(result_dir, f"integration_results_{param_key}.csv")
    
    # ========== 绘制去噪结果 ==========
    if os.path.exists(denoise_file):
        df_denoise = pd.read_csv(denoise_file)
        param_values_denoise = df_denoise[param_col].values
        ari_values_denoise = df_denoise['ARI'].values
        sort_idx = np.argsort(param_values_denoise)
        param_values_denoise = param_values_denoise[sort_idx]
        ari_values_denoise = ari_values_denoise[sort_idx]
        
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.plot(param_values_denoise, ari_values_denoise, marker='o', linewidth=2, 
               markersize=8, color='#2E86AB', linestyle='-')
        ax.set_xlabel(param_label, fontsize=12, fontweight='bold')
        ax.set_ylabel('ARI', fontsize=12, fontweight='bold')
        ax.set_title(f'Denoise Sensitivity Analysis: {param_label}', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        
        y_min = max(0.80, ari_values_denoise.min() - 0.02)
        y_max = min(1.0, ari_values_denoise.max() + 0.02)
        ax.set_ylim([y_min, y_max])
        
        plt.tight_layout()
        output_path = os.path.join(figs_dir, f"denoise_ari_{param_key}.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"去噪图片已保存到: {output_path}")
        plt.close()
    
    # ========== 绘制整合结果 ==========
    if os.path.exists(integration_file):
        df_integration = pd.read_csv(integration_file)
        param_values_integration = df_integration[param_col].values
        ari_values_integration = df_integration['ARI'].values
        sort_idx = np.argsort(param_values_integration)
        param_values_integration = param_values_integration[sort_idx]
        ari_values_integration = ari_values_integration[sort_idx]
        
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.plot(param_values_integration, ari_values_integration, marker='s', linewidth=2, 
               markersize=8, color='#2E86AB', linestyle='-')
        ax.set_xlabel(param_label, fontsize=12, fontweight='bold')
        ax.set_ylabel('ARI', fontsize=12, fontweight='bold')
        ax.set_title(f'Integration Sensitivity Analysis: {param_label}', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        
        y_min = max(0.80, ari_values_integration.min() - 0.02)
        y_max = min(1.0, ari_values_integration.max() + 0.02)
        ax.set_ylim([y_min, y_max])
        
        plt.tight_layout()
        output_path = os.path.join(figs_dir, f"integration_ari_{param_key}.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"整合图片已保存到: {output_path}")
        plt.close()

print("\n所有图片已生成完成！")


去噪图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\denoise_ari_spatial_k.png
整合图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\integration_ari_spatial_k.png
去噪图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\denoise_ari_feature_k.png
整合图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\integration_ari_feature_k.png
去噪图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\denoise_ari_diffusion_step.png
整合图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\integration_ari_diffusion_step.png
去噪图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\denoise_ari_alpha.png
整合图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results\figs\integration_ari_alpha.png
去噪图片已保存到: E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_anal

In [13]:
from codes.get_graph import construct_neighbor_graph,adjacent_matrix_preprocessing
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

adata_omics1, adata_omics2 = construct_neighbor_graph(adata_omics1, adata_omics2)

adj = adjacent_matrix_preprocessing(adata_omics1, adata_omics2)
adj_spatial_omics1 = adj['adj_spatial_omics1'].to(device)
adj_spatial_omics2 = adj['adj_spatial_omics2'].to(device)

In [14]:
ae_model = encoder_ZINB(
    adata=adata_omics1,
    device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu'),
    epochs=300, 
    dim_output=128, # 256
)

adata_omics1.obsm['emb_ZINB'], adj_mat = ae_model.train()

100%|██████████| 300/300 [00:02<00:00, 149.81it/s]

Optimization finished


In [15]:
seed = 2024
seed_everything(seed)

from codes_v2.AutoEncoder import train_model
train_model(adata_omics1, adata_omics2, adj_spatial_omics1, adj_spatial_omics2, epochs=200)

Epoch [100/200], Loss RNA: 1.6032, Loss Protein: 0.6135
Epoch [200/200], Loss RNA: 1.3810, Loss Protein: 0.5124
Training complete!
Latent representations have been successfully added to adata_omics.obsm!


## Denoise phase
Before the diffusion process, we need to align the two modality embeddings according to the spatial coordinates.

In [20]:
slices_omics1_spatial = adata_omics1.obsm['spatial']
slices_omics2_spatial = adata_omics2.obsm['spatial']

emb_latent_omics1 = adata_omics1.obsm['emb_ZINB']

emb_latent_omics2 = adata_omics2.obsm['emb_latent_omics2']

import pandas as pd

df_omics1 = pd.DataFrame(emb_latent_omics1, index=[tuple(coord) for coord in slices_omics1_spatial])
df_omics2 = pd.DataFrame(emb_latent_omics2, index=[tuple(coord) for coord in slices_omics2_spatial])

df_omics2_aligned = df_omics2.reindex(df_omics1.index)

aligned_emb_latent_omics1 = df_omics1.to_numpy()
aligned_emb_latent_omics2 = df_omics2_aligned.to_numpy()

print(aligned_emb_latent_omics1.shape)
print(aligned_emb_latent_omics2.shape)

(1296, 128)
(1296, 64)


In [21]:
class ConditionalDiffusionDataset():
    def __init__(self, adata_omics1, adata_omics2):
        self.adata_omics1 = adata_omics1
        self.adata_omics2 = adata_omics2

        self.st_sample = torch.tensor(self.adata_omics1, dtype=torch.float32)
        self.con_sample = torch.tensor(self.adata_omics2, dtype=torch.float32)
        self.con_data = torch.tensor(self.adata_omics2, dtype=torch.float32)

    def __len__(self):
        return len(self.adata_omics1)

    def __getitem__(self, idx):
        return self.st_sample[idx], self.con_sample[idx], self.con_data

In [ ]:
dataset = ConditionalDiffusionDataset(aligned_emb_latent_omics1, aligned_emb_latent_omics2) # denoise the first modality, condition on the second modality.

seed = 42
seed_everything(seed)

com_mtx = run_diff(
    dataset,

    k=3,
    batch_size=512,
    hidden_size=512,
    learning_rate=1e-3,

    num_epoch=1000,
    diffusion_step=800,

    depth=6,
    head=16,

    device='cuda:0',
    classes=6,  
    patience=40,
    bias=1 
)

*******************  Fold 1  (total 3 folds)  ********************


 39%|█████████▊               | 391/1000 [02:02<03:11,  3.18it/s, noise loss:0.0232200, lr:1.00e-06]


Early stop(patience:40)!!


time: 0: 100%|██████████| 800/800 [01:00<00:00, 13.18it/s]  


*******************  Fold 2  (total 3 folds)  ********************


...

time: 0: 100%|██████████| 800/800 [01:03<00:00, 12.57it/s]  


*******************  Fold 3  (total 3 folds)  ********************


 19%|████▊                    | 193/1000 [01:02<04:21,  3.09it/s, noise loss:0.0432099, lr:1.00e-04]


Early stop(patience:40)!!


time: 0: 100%|██████████| 800/800 [01:03<00:00, 12.52it/s]  


In [22]:
adata_omics1.obsm['denoise_emb'] = com_mtx

In [ ]:
from codes.preprocess1 import clustering

tool = 'mclust' # mclust, leiden, and louvain
clustering(adata_omics1, key='denoise_emb', add_key='Denoise', n_clusters=5, end=1.2, method=tool, use_pca=True)

R[write to console]:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.1.1
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%


In [ ]:
ground_truth = adata_omics1.obs['ground_truth']
denoise = adata_omics1.obs['Denoise']

ari_score_denoise = adjusted_rand_score(ground_truth, denoise)
ari_score_denoise

In [ ]:

import pandas as pd
from sklearn.metrics import normalized_mutual_info_score, mutual_info_score, adjusted_mutual_info_score
from sklearn.metrics import v_measure_score, homogeneity_score, completeness_score
from sklearn.metrics import adjusted_rand_score, fowlkes_mallows_score

GT_list = adata_omics2.obs['ground_truth']
Our_list = adata_omics2.obs['Denoise']

# 计算指标
Our_mutual_info = mutual_info_score(GT_list, Our_list)
Our_nmi = normalized_mutual_info_score(GT_list, Our_list)
Our_ami = adjusted_mutual_info_score(GT_list, Our_list)
Our_V = v_measure_score(GT_list, Our_list)
Our_homogeneity = homogeneity_score(GT_list, Our_list)
Our_completeness = completeness_score(GT_list, Our_list)
Our_ari = adjusted_rand_score(GT_list, Our_list)
Our_fmi = fowlkes_mallows_score(GT_list, Our_list)

# 打印结果
print('Ours\n')
print(f"Mutual Information: {Our_mutual_info:.6f}")
print(f"(NMI): {Our_nmi:.6f}")
print(f"(AMI): {Our_ami:.6f}")
print(f"V-measure: {Our_V:.6f}")
print(f"Homogeneity: {Our_homogeneity:.6f}")
print(f"Completeness: {Our_completeness:.6f}")
print(f"(ARI): {Our_ari:.6f}")
print(f"(FMI): {Our_fmi:.6f}")

# 将结果保存到 DataFrame
results = {
    'Method': ['ADT_denoised'],
    'Mutual Information': [Our_mutual_info],
    'NMI': [Our_nmi],
    'AMI': [Our_ami],
    'V-measure': [Our_V],
    'Homogeneity': [Our_homogeneity],
    'Completeness': [Our_completeness],
    'ARI': [Our_ari],
    'FMI': [Our_fmi]
}

df_results = pd.DataFrame(results)

# 将结果保存到 CSV 文件
csv_file = f"E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results/result.csv"
df_results.to_csv(csv_file, mode='a', header=not pd.io.common.file_exists(csv_file), index=False)

print(f"Results saved to {csv_file}")

Ours

Mutual Information: 1.293171
(NMI): 0.841837
(AMI): 0.841195
V-measure: 0.841837
Homogeneity: 0.835692
Completeness: 0.848073
(ARI): 0.846624
(FMI): 0.882368
Results saved to E:/yan0/ours/CANDIES_v2/sensitivity_analysis/results/lower_quality_as_condition/result.csv


## Integration phase

In [31]:
from codes.integration import *

In [32]:
adata1 = adata_omics1.copy()
adata2 = adata_omics2.copy()

In [ ]:
data_type = '10x'
adata1.obsm['feat'] = adata1.obsm['denoise_emb']
adata2.obsm['feat'] = adata2.obsm['emb_latent_omics2']
adata1, adata2 = construct_neighbor_graph(adata1, adata2)

In [34]:
adj = adjacent_matrix_preprocessing(adata1, adata2)
adj_spatial_omics1 = adj['adj_spatial_omics1'].to(device)
adj_spatial_omics2 = adj['adj_spatial_omics2'].to(device)
adj_feature_omics1 = adj['adj_feature_omics1'].to(device)
adj_feature_omics2 = adj['adj_feature_omics2'].to(device)

In [ ]:
features_omics1 = torch.FloatTensor(adata1.obsm['feat'].copy()).to(device)
features_omics2 = torch.FloatTensor(adata2.obsm['feat'].copy()).to(device)

seed = 2025
seed_everything(seed)

result = train_and_infer(
    features_omics1=features_omics1,
    features_omics2=features_omics2,
    adj_spatial_omics1=adj_spatial_omics1,
    adj_feature_omics1=adj_feature_omics1,
    adj_spatial_omics2=adj_spatial_omics2,
    adj_feature_omics2=adj_feature_omics2,
    device=device,
    epochs=100 # 300
)

Training Progress: 100%|██████████| 100/100 [00:13<00:00,  7.67it/s, Total Loss=16.9, Contrastive Loss (Spatial)=5.23, Contrastive Loss (Feature)=6.7, Reconstruction Loss 1=2.06, Reconstruction Loss 2=0.452, Reconstruction Loss 3=2.09, Reconstruction Loss 4=0.38] 


In [36]:
adata = adata1.copy()
adata.obsm['CANDIES'] = result['emb_latent_combined'].detach().cpu().numpy().copy()

In [37]:
tool = 'mclust'
clustering(adata, key='CANDIES', add_key='CANDIES', n_clusters=5, end=1, method=tool, use_pca=True)

fitting ...
  |======================================================================| 100%


In [38]:
from sklearn.metrics import adjusted_rand_score

# 提取 ground_truth 和 AE 列
ground_truth = adata.obs['ground_truth']
CANDIES = adata.obs['CANDIES']

# 计算 ARI
ari_score_CANDIES = adjusted_rand_score(ground_truth, CANDIES)

print(f"Adjusted Rand Index (ARI) between ground_truth and AE: {ari_score_CANDIES:.4f}")

Adjusted Rand Index (ARI) between ground_truth and AE: 0.9175


In [ ]:

import pandas as pd
from sklearn.metrics import normalized_mutual_info_score, mutual_info_score, adjusted_mutual_info_score
from sklearn.metrics import v_measure_score, homogeneity_score, completeness_score
from sklearn.metrics import adjusted_rand_score, fowlkes_mallows_score

GT_list = adata.obs['ground_truth']
Our_list = adata.obs['CANDIES']

# 计算指标
Our_mutual_info = mutual_info_score(GT_list, Our_list)
Our_nmi = normalized_mutual_info_score(GT_list, Our_list)
Our_ami = adjusted_mutual_info_score(GT_list, Our_list)
Our_V = v_measure_score(GT_list, Our_list)
Our_homogeneity = homogeneity_score(GT_list, Our_list)
Our_completeness = completeness_score(GT_list, Our_list)
Our_ari = adjusted_rand_score(GT_list, Our_list)
Our_fmi = fowlkes_mallows_score(GT_list, Our_list)

# 打印结果
print('Ours\n')
print(f"Mutual Information: {Our_mutual_info:.6f}")
print(f"(NMI): {Our_nmi:.6f}")
print(f"(AMI): {Our_ami:.6f}")
print(f"V-measure: {Our_V:.6f}")
print(f"Homogeneity: {Our_homogeneity:.6f}")
print(f"Completeness: {Our_completeness:.6f}")
print(f"(ARI): {Our_ari:.6f}")
print(f"(FMI): {Our_fmi:.6f}")

# 将结果保存到 DataFrame
results = {
    'Method': ['CANDIES(ADT_denoise)'],
    'Mutual Information': [Our_mutual_info],
    'NMI': [Our_nmi],
    'AMI': [Our_ami],
    'V-measure': [Our_V],
    'Homogeneity': [Our_homogeneity],
    'Completeness': [Our_completeness],
    'ARI': [Our_ari],
    'FMI': [Our_fmi]
}

df_results = pd.DataFrame(results)

# 将结果保存到 CSV 文件
csv_file = f"E:/yan0/ours/CANDIES_v2/sensitivity_analysis/parameters_analysis/results/result.csv"
df_results.to_csv(csv_file, mode='a', header=not pd.io.common.file_exists(csv_file), index=False)

print(f"Results saved to {csv_file}")

Ours

Mutual Information: 1.401326
(NMI): 0.905157
(AMI): 0.904775
V-measure: 0.905157
Homogeneity: 0.905585
Completeness: 0.904729
(ARI): 0.917541
(FMI): 0.936166
Results saved to E:/yan0/ours/CANDIES_v2/sensitivity_analysis/results/lower_quality_as_condition/result.csv
